In [ ]:
#!/usr/bin/env python3
"""
Gmail(@campus.ouj.ac.jp)用 IMAP/SMTP接続スクリプト
このセル単体で実行可能
"""

import imaplib
import smtplib
import ssl
import getpass
import email
from email.header import decode_header
from email.message import EmailMessage


# ===== 設定 =====
IMAP_HOST = "imap.gmail.com"
IMAP_PORT = 993

SMTP_HOST = "smtp.gmail.com"
SMTP_PORT = 465

# あなたのメールアドレスを設定してください
EMAIL_ADDRESS = "your_email@example.com"


def decode_mime_header(value):
    """MIMEエンコードされたヘッダを読みやすい文字列に復号"""
    if not value:
        return ""
    decoded_parts = decode_header(value)
    decoded_str = ""
    for part, enc in decoded_parts:
        if isinstance(part, bytes):
            decoded_str += part.decode(enc or "utf-8", errors="replace")
        else:
            decoded_str += part
    return decoded_str


def test_imap_connection(email_address, app_password, limit=5):
    """IMAPで受信メールを取得してテスト"""
    print("=" * 60)
    print("IMAP接続テスト開始")
    print("=" * 60)
    
    context = ssl.create_default_context()
    
    try:
        # IMAP接続
        with imaplib.IMAP4_SSL(IMAP_HOST, IMAP_PORT, ssl_context=context) as imap:
            print(f"✓ サーバー接続成功: {IMAP_HOST}:{IMAP_PORT}")
            
            # ログイン
            print(f"ログイン中: {email_address}")
            imap.login(email_address, app_password)
            print("✓ 認証成功")
            
            # INBOX選択
            status, messages = imap.select("INBOX")
            if status == "OK":
                total_messages = int(messages[0])
                print(f"✓ INBOX選択成功 (総メール数: {total_messages}件)")
            
            # メール検索
            typ, data = imap.search(None, "ALL")
            if typ != "OK":
                print("✗ メール検索失敗")
                return False
            
            msg_ids = data[0].split()
            if not msg_ids:
                print("メールが見つかりませんでした")
                return True
            
            # 直近のメールを表示
            recent_ids = msg_ids[-limit:]
            print(f"\n直近 {len(recent_ids)} 件のメール:")
            print("-" * 60)
            
            for idx in reversed(recent_ids):
                typ, msg_data = imap.fetch(idx, "(RFC822)")
                if typ == "OK" and msg_data and msg_data[0]:
                    raw_email = msg_data[0][1]
                    msg = email.message_from_bytes(raw_email)
                    
                    subject = decode_mime_header(msg.get("Subject", ""))
                    from_ = decode_mime_header(msg.get("From", ""))
                    date_ = decode_mime_header(msg.get("Date", ""))
                    
                    print(f"\n[{idx.decode('ascii', errors='ignore')}]")
                    print(f"  From: {from_}")
                    print(f"  Date: {date_}")
                    print(f"  Subject: {subject}")
            
            print("\n" + "=" * 60)
            print("✓ IMAP接続テスト成功")
            print("=" * 60)
            return True
            
    except imaplib.IMAP4.error as e:
        print(f"\n✗ IMAP認証エラー: {e}")
        print("アプリパスワードが正しいか確認してください")
        return False
    except Exception as e:
        print(f"\n✗ IMAPエラー: {type(e).__name__}: {e}")
        return False


def test_smtp_connection(email_address, app_password):
    """SMTPで自分宛にテストメールを送信"""
    print("\n" + "=" * 60)
    print("SMTP送信テスト開始")
    print("=" * 60)
    
    context = ssl.create_default_context()
    
    try:
        # メッセージ作成
        msg = EmailMessage()
        msg["Subject"] = "【テスト】IMAP/SMTP接続確認"
        msg["From"] = email_address
        msg["To"] = email_address
        msg.set_content(
            "このメールはIMAP/SMTP接続テストです。\n\n"
            f"送信日時: {email.utils.formatdate(localtime=True)}\n"
            "IMAPとSMTPの接続が正常に動作しています。\n"
        )
        
        # SMTP接続 (SSL/TLS)
        with smtplib.SMTP_SSL(SMTP_HOST, SMTP_PORT, context=context) as server:
            print(f"✓ サーバー接続成功: {SMTP_HOST}:{SMTP_PORT}")
            
            # ログイン
            print(f"ログイン中: {email_address}")
            server.login(email_address, app_password)
            print("✓ 認証成功")
            
            # メール送信
            server.send_message(msg)
            print(f"✓ メール送信成功: {email_address} → {email_address}")
        
        print("\n" + "=" * 60)
        print("✓ SMTP送信テスト成功")
        print("=" * 60)
        return True
        
    except smtplib.SMTPAuthenticationError as e:
        print(f"\n✗ SMTP認証エラー: {e}")
        print("アプリパスワードが正しいか確認してください")
        return False
    except Exception as e:
        print(f"\n✗ SMTPエラー: {type(e).__name__}: {e}")
        return False


def main():
    """メイン処理"""
    print("\n" + "=" * 60)
    print("Gmail IMAP/SMTP 接続テスト")
    print("=" * 60)
    print(f"メールアドレス: {EMAIL_ADDRESS}")
    print("\n【重要】")
    print("Googleアカウントのアプリパスワードを使用してください。")
    print("通常のパスワードでは接続できません。")
    print("\nアプリパスワードの生成方法:")
    print("1. Googleアカウント → セキュリティ → 2段階認証を有効化")
    print("2. セキュリティ → アプリパスワード")
    print("3. アプリ:メール、デバイス:Windowsパソコン を選択")
    print("4. 生成された16文字のパスワード(スペースなし)を入力")
    print("=" * 60 + "\n")
    
    # アプリパスワード入力
    app_password = getpass.getpass("アプリパスワードを入力してください: ")
    
    if not app_password:
        print("パスワードが入力されていません")
        return
    
    # IMAP接続テスト
    imap_success = test_imap_connection(EMAIL_ADDRESS, app_password, limit=5)
    
    # SMTP送信テスト
    if imap_success:
        print("\n続けてSMTP送信テストを実行しますか? (y/n): ", end="")
        try:
            response = input().lower()
            if response == 'y':
                smtp_success = test_smtp_connection(EMAIL_ADDRESS, app_password)
            else:
                print("SMTP送信テストをスキップしました")
        except:
            print("\n処理を中断しました")
    
    print("\n" + "=" * 60)
    print("テスト完了")
    print("=" * 60)


# このセルを実行
if __name__ == "__main__":
    main()



Gmail IMAP/SMTP 接続テスト
メールアドレス: kentaro199910@gmail.com

【重要】
Googleアカウントのアプリパスワードを使用してください。
通常のパスワードでは接続できません。

アプリパスワードの生成方法:
1. Googleアカウント → セキュリティ → 2段階認証を有効化
2. セキュリティ → アプリパスワード
3. アプリ:メール、デバイス:Windowsパソコン を選択
4. 生成された16文字のパスワード(スペースなし)を入力

IMAP接続テスト開始
IMAP接続テスト開始
✓ サーバー接続成功: imap.gmail.com:993
ログイン中: kentaro199910@gmail.com
✓ サーバー接続成功: imap.gmail.com:993
ログイン中: kentaro199910@gmail.com
✓ 認証成功
✓ 認証成功
✓ INBOX選択成功 (総メール数: 16991件)
✓ INBOX選択成功 (総メール数: 16991件)

直近 5 件のメール:
------------------------------------------------------------

直近 5 件のメール:
------------------------------------------------------------

[16991]
  From: Bob Donovan <notifications@github.com>
  Date: Thu, 04 Dec 2025 14:21:36 -0800
  Subject: Re: [Unity-Technologies/ml-agents] Release/4.0.1 (PR #6265)

[16991]
  From: Bob Donovan <notifications@github.com>
  Date: Thu, 04 Dec 2025 14:21:36 -0800
  Subject: Re: [Unity-Technologies/ml-agents] Release/4.0.1 (PR #6265)

[16990]
  From: Bob Donovan <notifications@github.c